# Phase 1 — Data profiling and EDA

OpenML Credit Card Fraud ID 1597. Course topics: mean, median, P99, skew, missing values, duplicates, PCA, leakage, PySpark. Data must be present locally at `data/raw/creditcard.csv`. No model training, SMOTE, splitting, scaling or row deletion in this notebook.

Run from repo root, after `pip install -e '.[data,ml,dev]'`. Source teaching materials: `bigdata_analytics_preprocess_data.pdf`, `bigdata_analytics_p1.ipynb`, and `BDA_PySpark.ipynb` in BDA_Course Drive.

## 1. Load and validate source

The tested package validates the 31-column contract and generates a local SHA-256 manifest. Expected original counts are only references until you execute this notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from fraud_detection.ingestion.profiling import run_profile

DATA = Path('data/raw/creditcard.csv')
OUTPUT = Path('artifacts/phase_01')
if not DATA.is_file():
    raise FileNotFoundError('Download OpenML 1597 CSV to data/raw/creditcard.csv')
df = pd.read_csv(DATA)
report = run_profile(DATA, OUTPUT)
print('rows', report['rows'], 'fraud count', report['class_counts']['1'], 'valid', report['valid'])
display(df.head())

## 2. Schema, data dictionary, and quality findings

`Time` is relative seconds, `Amount` is the transaction value, `Class` is binary label, and V1–V28 are anonymized PCA components. Duplicate rows are retained, not automatically discarded.

In [ ]:
display(pd.read_csv(OUTPUT / 'column_profile.csv'))
display(pd.DataFrame(report['findings']))
print('Duplicate rows:', report['duplicate_rows'])
print('Duplicates by label:', report['duplicates_by_class'])
print('Missing values:', report['null_count'])

## 3. Descriptive statistics — mean, median, P99, skewness

These terms match the course statistics notebook. Right-skewness is expected to be investigated, not assumed.

In [ ]:
display(df['Amount'].describe(percentiles=[0.25,0.5,0.75,0.95,0.99]))
print('Skew:', df['Amount'].skew())
fig, ax = plt.subplots(figsize=(9,4))
ax.hist(df['Amount'].dropna(), bins=80)
ax.axvline(df['Amount'].median(), linestyle='--', label='Median')
ax.axvline(df['Amount'].quantile(.99), linestyle=':', label='P99')
ax.set(xlabel='Amount', ylabel='Count', title='Amount distribution from local CSV')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Class imbalance and majority-class accuracy paradox

Count the exact minority prevalence before deciding which metrics will be appropriate in later phases. Do not resample.

In [ ]:
counts = df['Class'].value_counts().sort_index()
display(pd.DataFrame({'count': counts, 'pct': counts / len(df) * 100}))
ax = counts.plot.bar(title='Original class distribution', ylabel='Transactions')
plt.tight_layout()
plt.show()
print('Accuracy of majority-only prediction:', counts.max()/len(df))

## 5. Conditional statistics and PCA scatter

PCA features do not reveal original business meanings. Outliers may be important fraud signals.

In [ ]:
display(df.groupby('Class')[['Amount','Time']].agg(['count','mean','median','min','max']))
sample = df.sample(n=min(len(df),20000), random_state=42)
fig, ax = plt.subplots(figsize=(8,5))
for label, group in sample.groupby('Class'):
    ax.scatter(group['V1'],group['V2'],s=5,alpha=.35,label=f'Class {label}')
ax.set(xlabel='V1',ylabel='V2',title='Anonymized PCA features — sampled')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Leakage-risk notes and review

Do not fit scalers or PCA before splitting; resample only training folds; guard exact duplicate overlap among splits; use validation for threshold tuning and keep test untouched. The actual split is reserved for Phase 2.

Output `profile.json`, `column_profile.csv`, and `dataset_manifest.json` into the ignored local artifacts folder. Review findings and document quality decisions before Phase 2.

In [ ]:
print('Local outputs:')
for path in sorted(OUTPUT.glob('*')):
    print(path)
print('Phase 1 finished: exploration only, no ML or benchmark.')